# 제출 2. 규제를 강화한 LightGBM

- 근거: `exp04`에서 OOF AUC가 가장 높았던 설정 (4-6, 0.95909 / baseline 0.95866)
- 변수는 원본 그대로, 파라미터만 변경
- 제출 1: LightGBM 기본 설정, 리더보드 0.95802

## 환경설정

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

## 데이터 경로 탐색

In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)

# 크로스플랫폼 경로 자동 해결

def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data


## 데이터 로드 및 전처리

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]

In [4]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 교차검증 함수

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}
N_FOLDS = 5


def run_cv(X, X_test, params=LGB_PARAMS):
    """Stratified 5-fold LightGBM; returns the OOF AUC and the averaged test prediction."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration}")

    return roc_auc_score(y, oof_pred), test_pred

## 최종 모델 학습 (실험 4-6 설정)

In [6]:
BEST_PARAMS = {**LGB_PARAMS, "feature_fraction": 0.6, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 30}
auc, test_pred = run_cv(X, X_test, BEST_PARAMS)
print(f"\nOOF AUC = {auc:.5f}")

Fold 1: AUC = 0.95932   trees = 781


Fold 2: AUC = 0.95788   trees = 778


Fold 3: AUC = 0.95972   trees = 564


Fold 4: AUC = 0.95926   trees = 735


Fold 5: AUC = 0.95934   trees = 388

OOF AUC = 0.95909


## 제출 파일 생성 및 정합성 검증

In [7]:
SUBMISSION_PATH = Path("../submissions/submission_lgb_reg.csv")
SUBMISSION_PATH.parent.mkdir(exist_ok=True)

pred_df = pd.DataFrame({"id": test["id"].values, TARGET: test_pred})

assert pred_df["id"].is_unique
assert set(pred_df["id"]) == set(sample["id"])

submission = sample[["id"]].merge(pred_df, on="id", how="left", validate="one_to_one")

assert len(submission) == len(sample)
assert (submission["id"].values == sample["id"].values).all()
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

submission.to_csv(SUBMISSION_PATH, index=False)
print("saved:", SUBMISSION_PATH.name, submission.shape)
submission.head()

saved: submission_lgb_reg.csv (299844, 2)


,id,satisfaction
0,699635,0.241480
1,699636,0.896519
2,699637,0.898264
3,699638,0.065952
4,699639,0.054417


In [8]:
print(f"전체 실행 시간: {time.time() - START_TIME:.0f}초")

전체 실행 시간: 84초
